# KNN Independent BP/RP Truncation Sweep

This notebook evaluates how binary-classification performance changes when the BP and RP Gaia XP coefficient arms are truncated independently. Each point on the grid uses the first `K_bp` BP coefficients and the first `K_rp` RP coefficients, then summarizes performance across repeated stratified train/validation/test splits.

The primary analysis surface is the `(K_bp, K_rp)` grid for repeat-level metrics. The reporting cells produce a common visualization design across models: mean metric value, 95% confidence interval for the mean, repeat count, and ranked high-performing grid regions.


## Configuration and experiment state


In [ ]:
import os, time, threading
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from contextlib import contextmanager

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, log_loss
from sklearn.neighbors import KNeighborsClassifier

from joblib import Parallel, delayed
import joblib


# Resolve project paths from the repository directory rather than from the kernel launch directory.
def find_project_root(start=None):
    marker_files = [
        "out_data/gaia_dr3_xp_c110_l2_binary.csv",
        "preparations/VOSA_labels_training.csv",
        "README.md",
    ]
    configured_root = os.environ.get("ASTROPHYSICS_PROJECT_ROOT")
    search_roots = []
    if configured_root:
        search_roots.append(Path(configured_root).expanduser())

    start = Path.cwd() if start is None else Path(start)
    for root in [start, *start.parents]:
        packaged_root = root / "prep"
        if packaged_root.exists():
            search_roots.append(packaged_root)
        search_roots.append(root)

    desktop_project = Path.home() / "Desktop" / "Astrophysics"
    if (desktop_project / "prep").exists():
        search_roots.append(desktop_project / "prep")
    if desktop_project.exists():
        search_roots.append(desktop_project)

    seen = set()
    for root in search_roots:
        root = root.resolve()
        if root in seen:
            continue
        seen.add(root)
        if any((root / marker).exists() for marker in marker_files):
            return root

    raise FileNotFoundError(
        "Could not locate the project root. Start Jupyter from the prep directory "
        "or set ASTROPHYSICS_PROJECT_ROOT to the directory that contains this truncation bundle."
    )


def resolve_project_path(path):
    path = Path(path).expanduser()
    return path if path.is_absolute() else PROJECT_ROOT / path

PROJECT_ROOT = find_project_root()

# Paths and output locations
CSV_PATH = resolve_project_path(globals().get("CSV_PATH", PROJECT_ROOT / "out_data" / "gaia_dr3_xp_c110_l2_binary.csv"))
OUT_DIR = resolve_project_path(globals().get("OUT_DIR", PROJECT_ROOT / "55x55_experiments" / "knn_55x55_truncation_out"))

# BP/RP grid shards are stored separately from the one-dimensional truncation outputs.
BP_RP_GRID_RUNS_DIR = OUT_DIR / "runs_bp_rp_grid"
BP_RP_GRID_RUNS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_LABEL = "KNN"
RAW_OUTPUT_BASENAME = "truncation_knn_bp_rp_grid_raw.csv"
RAW_PARQUET = OUT_DIR / RAW_OUTPUT_BASENAME.replace(".csv", ".parquet")
RAW_CSV = OUT_DIR / RAW_OUTPUT_BASENAME
REPORT_DIR = OUT_DIR / "analysis_report"
FIGURE_DIR = REPORT_DIR / "figures"
for directory in [REPORT_DIR, FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Experiment configuration
N_REPEATS = int(globals().get("N_REPEATS", 100))
TEST_FRAC = float(globals().get("TEST_FRAC", 0.15))
VAL_FRAC  = float(globals().get("VAL_FRAC", 0.15))
VAL_FRAC_OF_REMAIN = float(globals().get("VAL_FRAC_OF_REMAIN", VAL_FRAC / (1.0 - TEST_FRAC)))

BASE_SEED = int(globals().get("BASE_SEED", 1234))
THR_GRID_SIZE = int(globals().get("THR_GRID_SIZE", 800))

# KNN params from KNN_truncation.ipynb
KNN_PARAMS_BASE = dict(
    n_neighbors=25,
    weights="distance",
    metric="minkowski",
    p=2,
    n_jobs=-1,
)

# BP/RP grid full 2D grid
K_GRID = list(range(1, 56))
K_BP_LIST = list(K_GRID)
K_RP_LIST = list(K_GRID)

# Parallelization (repeat-level). Use a conservative default for stability.
N_JOBS = int(globals().get("N_JOBS", max(1, min(6, (os.cpu_count() or 4) - 1))))
CHECKPOINT_EVERY = int(globals().get("CHECKPOINT_EVERY", 24))

print("CSV_PATH:", CSV_PATH)
print("OUT_DIR:", OUT_DIR)
print("BP_RP_GRID_RUNS_DIR:", BP_RP_GRID_RUNS_DIR)
print("Repeats:", N_REPEATS, " | K pairs:", len(K_BP_LIST) * len(K_RP_LIST), " | N_JOBS:", N_JOBS)
print("Checkpoint every:", CHECKPOINT_EVERY, "pairs")


## Load Gaia XP coefficient dataset


In [ ]:
# Load the coefficient dataset once per kernel session.
if "df" not in globals():
    df = pd.read_csv(CSV_PATH)
if "y" not in df.columns:
    raise KeyError("CSV must contain 'y'")
df["y"] = df["y"].astype(int)

C110 = [f"c{i:03d}" for i in range(110)]
missing = [c for c in C110 if c not in df.columns]
if missing:
    raise KeyError(f"Missing c110 columns, e.g. {missing[:6]}")

# c000..c054 = BP, c055..c109 = RP
BP = [f"c{i:03d}" for i in range(55)]
RP = [f"c{i:03d}" for i in range(55, 110)]

Xbp = df[BP].to_numpy(np.float32)
Xrp = df[RP].to_numpy(np.float32)
y_all = df["y"].to_numpy(np.int32)

print("Xbp:", Xbp.shape, "Xrp:", Xrp.shape, "y:", y_all.shape)


## Create stratified train, validation, and test splits


In [ ]:
def make_splits(y, n_repeats, base_seed):
    splitter = StratifiedShuffleSplit(n_splits=n_repeats, test_size=TEST_FRAC, random_state=base_seed)
    idx_all = np.arange(len(y))
    splits = []
    for r, (trainval_rel, test_rel) in enumerate(splitter.split(np.zeros(len(y)), y)):
        idx_trainval = idx_all[trainval_rel]
        idx_test = idx_all[test_rel]

        y_tv = y[idx_trainval]
        splitter2 = StratifiedShuffleSplit(n_splits=1, test_size=VAL_FRAC_OF_REMAIN, random_state=base_seed + 1000 + r)
        tr_rel2, va_rel2 = next(splitter2.split(np.zeros(len(y_tv)), y_tv))

        idx_train = idx_trainval[tr_rel2]
        idx_val   = idx_trainval[va_rel2]
        splits.append(dict(repeat=r, train_idx=idx_train, val_idx=idx_val, test_idx=idx_test))
    return splits

splits = make_splits(y_all, N_REPEATS, BASE_SEED)
print("splits:", len(splits), "sizes:", {k: len(splits[0][k]) for k in ["train_idx","val_idx","test_idx"]})


## Metric and threshold helpers


In [ ]:
def prob_metrics(y_true, p):
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p).astype(float)
    pc = np.clip(p, 1e-15, 1-1e-15)
    return dict(
        ROC_AUC=roc_auc_score(y_true, p) if len(np.unique(y_true)) > 1 else np.nan,
        PR_AUC=average_precision_score(y_true, p) if len(np.unique(y_true)) > 1 else np.nan,
        Brier=brier_score_loss(y_true, p),
        LogLoss=log_loss(y_true, pc, labels=[0,1]),
    )

def threshold_grid(p, n=800):
    lo, hi = float(np.min(p)), float(np.max(p))
    if not np.isfinite(lo) or not np.isfinite(hi) or lo == hi:
        return np.array([0.5], dtype=float)
    return np.linspace(lo, hi, int(n), dtype=float)

def choose_threshold_fastgrid(y_true, p, criterion="youden", n=800):
    y = np.asarray(y_true, dtype=np.int32)
    p = np.asarray(p, dtype=float)

    thr = threshold_grid(p, n)
    if thr.size == 1:
        return float(thr[0]), -np.inf

    order = np.argsort(-p)
    p_s = p[order]
    y_s = y[order]

    tp_cum = np.cumsum(y_s, dtype=np.int64)
    fp_cum = np.cumsum(1 - y_s, dtype=np.int64)

    P = int(tp_cum[-1])
    N = int(fp_cum[-1])
    if P == 0 or N == 0:
        return float(thr[len(thr)//2]), -np.inf

    k = np.searchsorted(-p_s, -thr, side="right")

    tp = np.where(k > 0, tp_cum[k - 1], 0)
    fp = np.where(k > 0, fp_cum[k - 1], 0)
    fn = P - tp
    tn = N - fp

    sens = tp / P
    spec = tn / N
    prec = np.divide(tp, tp + fp, out=np.full_like(tp, np.nan, dtype=float), where=(tp + fp) > 0)

    if criterion == "youden":
        score = sens + spec - 1.0
    elif criterion == "f1":
        denom = prec + sens
        score = np.divide(2 * prec * sens, denom, out=np.full_like(denom, -np.inf, dtype=float),
                          where=np.isfinite(denom) & (denom > 0))
    else:
        raise ValueError("criterion must be 'youden' or 'f1'")

    score2 = np.where(np.isfinite(score), score, -np.inf)
    j = int(np.argmax(score2))
    return float(thr[j]), float(score2[j])

def eval_threshold_metrics(y_true, p, thr):
    y = np.asarray(y_true, dtype=np.int32)
    yhat = (np.asarray(p, dtype=float) >= float(thr)).astype(np.int32)

    tp = int(np.sum((yhat == 1) & (y == 1)))
    tn = int(np.sum((yhat == 0) & (y == 0)))
    fp = int(np.sum((yhat == 1) & (y == 0)))
    fn = int(np.sum((yhat == 0) & (y == 1)))

    sens = tp / (tp + fn) if (tp + fn) else np.nan
    spec = tn / (tn + fp) if (tn + fp) else np.nan
    prec = tp / (tp + fp) if (tp + fp) else np.nan

    return dict(
        Precision=float(prec),
        Sensitivity=float(sens),
        Specificity=float(spec),
        tn=tn, fp=fp, fn=fn, tp=tp
    )

def evaluate(yv, pv, yt, pt):
    out = {f"test_{k}": float(v) for k, v in prob_metrics(yt, pt).items()}
    for crit in ["youden", "f1"]:
        thr, sc = choose_threshold_fastgrid(yv, pv, crit, THR_GRID_SIZE)
        met = eval_threshold_metrics(yt, pt, thr)
        out[f"{crit}_val_thr"] = float(thr)
        out[f"{crit}_val_score"] = float(sc)
        out[f"{crit}_test_Precision"] = float(met["Precision"])
        out[f"{crit}_test_Sensitivity"] = float(met["Sensitivity"])
        out[f"{crit}_test_Specificity"] = float(met["Specificity"])
        out[f"{crit}_test_tn"] = int(met["tn"]); out[f"{crit}_test_fp"] = int(met["fp"])
        out[f"{crit}_test_fn"] = int(met["fn"]); out[f"{crit}_test_tp"] = int(met["tp"])
    return out


## Repeat-level execution helpers


In [ ]:
def repeat_shard_path(rep: int) -> Path:
    return BP_RP_GRID_RUNS_DIR / f"bp_rp_grid_r{int(rep):03d}.parquet"

def repeat_tmp_path(rep: int) -> Path:
    return BP_RP_GRID_RUNS_DIR / f"bp_rp_grid_r{int(rep):03d}.tmp.parquet"

def expected_pair_count() -> int:
    return len(K_BP_LIST) * len(K_RP_LIST)

def _pair_count_in_file(path: Path) -> int:
    if not path.exists():
        return 0
    try:
        dfp = pd.read_parquet(path, columns=["K_bp", "K_rp"])
        if len(dfp) == 0:
            return 0
        return int(dfp.drop_duplicates(["K_bp", "K_rp"]).shape[0])
    except Exception:
        return 0

def done_rep(rep: int) -> bool:
    return done_pair_count(rep) >= expected_pair_count()

def done_pair_count(rep: int) -> int:
    out_path = repeat_shard_path(rep)
    tmp_path = repeat_tmp_path(rep)
    return max(_pair_count_in_file(out_path), _pair_count_in_file(tmp_path))

def update_pair_progress(n: int = 1):
    pbar = globals().get("PAIR_PBAR", None)
    if pbar is None:
        return
    lock = globals().get("PAIR_PBAR_LOCK", None)
    if lock is None:
        pbar.update(n)
    else:
        with lock:
            pbar.update(n)

def make_X_pair_scaled(Xbp_s, Xrp_s, K_bp: int, K_rp: int):
    return np.concatenate([Xbp_s[:, :int(K_bp)], Xrp_s[:, :int(K_rp)]], axis=1)

def run_one_repeat_bp_rp_grid(sp: dict):
    rep = int(sp["repeat"])
    out_path = repeat_shard_path(rep)
    tmp_path = repeat_tmp_path(rep)
    expected_pairs = expected_pair_count()

    tr = sp["train_idx"]; va = sp["val_idx"]; te = sp["test_idx"]

    ytr = y_all[tr]
    yva = y_all[va]
    yte = y_all[te]

    # Standardization is essential for KNN; compute on TRAIN once for full BP/RP and slice later
    bp_tr = Xbp[tr].astype(np.float64, copy=False)
    rp_tr = Xrp[tr].astype(np.float64, copy=False)

    mu_bp = bp_tr.mean(axis=0)
    sd_bp = bp_tr.std(axis=0, ddof=0)
    sd_bp = np.where(sd_bp > 0, sd_bp, 1.0)

    mu_rp = rp_tr.mean(axis=0)
    sd_rp = rp_tr.std(axis=0, ddof=0)
    sd_rp = np.where(sd_rp > 0, sd_rp, 1.0)

    Xbp_tr_s = (Xbp[tr].astype(np.float64, copy=False) - mu_bp) / sd_bp
    Xbp_va_s = (Xbp[va].astype(np.float64, copy=False) - mu_bp) / sd_bp
    Xbp_te_s = (Xbp[te].astype(np.float64, copy=False) - mu_bp) / sd_bp

    Xrp_tr_s = (Xrp[tr].astype(np.float64, copy=False) - mu_rp) / sd_rp
    Xrp_va_s = (Xrp[va].astype(np.float64, copy=False) - mu_rp) / sd_rp
    Xrp_te_s = (Xrp[te].astype(np.float64, copy=False) - mu_rp) / sd_rp

    params = dict(KNN_PARAMS_BASE)

    rows = []
    done_pairs = set()
    seed_frames = []
    if out_path.exists():
        try:
            seed_frames.append(pd.read_parquet(out_path))
        except Exception:
            out_path.unlink(missing_ok=True)
    if tmp_path.exists():
        try:
            seed_frames.append(pd.read_parquet(tmp_path))
        except Exception:
            tmp_path.unlink(missing_ok=True)

    if seed_frames:
        seed = pd.concat(seed_frames, ignore_index=True)
        if len(seed):
            seed["K_bp"] = seed["K_bp"].astype(int)
            seed["K_rp"] = seed["K_rp"].astype(int)
            seed = seed.drop_duplicates(["K_bp", "K_rp"], keep="last")
            rows = seed.to_dict("records")
            done_pairs = {(int(kb), int(kr)) for kb, kr in zip(seed["K_bp"], seed["K_rp"])}

    for K_bp in K_BP_LIST:
        for K_rp in K_RP_LIST:
            if (int(K_bp), int(K_rp)) in done_pairs:
                continue

            t0 = time.time()

            Xtr = make_X_pair_scaled(Xbp_tr_s, Xrp_tr_s, K_bp, K_rp)
            Xva = make_X_pair_scaled(Xbp_va_s, Xrp_va_s, K_bp, K_rp)
            Xte = make_X_pair_scaled(Xbp_te_s, Xrp_te_s, K_bp, K_rp)

            model = KNeighborsClassifier(**params)
            model.fit(Xtr, ytr)

            pv = model.predict_proba(Xva)[:, 1]
            pt = model.predict_proba(Xte)[:, 1]

            met = evaluate(yva, pv, yte, pt)
            dt = time.time() - t0

            rows.append(dict(
                run_id=f"KB{int(K_bp):02d}__KR{int(K_rp):02d}__r{rep:03d}",
                K_bp=int(K_bp),
                K_rp=int(K_rp),
                repeat=int(rep),
                runtime_s=float(dt),
                n_train=int(len(ytr)),
                n_val=int(len(yva)),
                n_test=int(len(yte)),
                test_pos_rate=float(np.mean(yte)),
                **met
            ))
            done_pairs.add((int(K_bp), int(K_rp)))
            update_pair_progress(1)

            if CHECKPOINT_EVERY > 0 and (len(done_pairs) % CHECKPOINT_EVERY == 0 or len(done_pairs) == expected_pairs):
                pd.DataFrame(rows).to_parquet(tmp_path, index=False)

    out = pd.DataFrame(rows)
    out.to_parquet(out_path, index=False)
    if tmp_path.exists():
        tmp_path.unlink()
    return str(out_path)


## Use precomputed results or run the BP/RP grid sweep


In [ ]:
# Execution controls for the BP/RP grid sweep.
USE_PRECOMPUTED_BP_RP_GRID_RESULTS = bool(globals().get("USE_PRECOMPUTED_BP_RP_GRID_RESULTS", True))
RUN_BP_RP_GRID_SWEEP = bool(globals().get("RUN_BP_RP_GRID_SWEEP", False))
ALLOW_PARTIAL_PRECOMPUTED_RESULTS = bool(globals().get("ALLOW_PARTIAL_PRECOMPUTED_RESULTS", False))
SHARD_PROGRESS_EVERY = int(globals().get("SHARD_PROGRESS_EVERY", 10))
WRITE_BP_RP_GRID_CSV = bool(globals().get("WRITE_BP_RP_GRID_CSV", False))

ALTERNATE_RESULT_PATHS = [
    Path("55x55_experiments/knn_55x55_truncation_out/truncation_knn_bp_rp_grid_raw.parquet"),
    Path("55x55_experiments/knn_55x55_truncation_out/truncation_knn_bp_rp_grid_raw.csv"),
]
BP_RP_GRID_CHUNK_DIRS = []
REQUIRED_RESULT_COLUMNS = {"K_bp", "K_rp", "repeat", "test_PR_AUC", "test_ROC_AUC", "test_LogLoss"}


def _resolve_bp_rp_grid_path(path: Path) -> Path:
    path = Path(path)
    return path if path.is_absolute() else PROJECT_ROOT / path


def expected_bp_rp_grid_rows() -> int:
    return len(K_BP_LIST) * len(K_RP_LIST) * int(N_REPEATS)


def _read_bp_rp_grid_result(path: Path, columns=None) -> pd.DataFrame:
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path, columns=columns)
    if columns is None:
        return pd.read_csv(path)
    return pd.read_csv(path, usecols=columns)


def _bp_rp_grid_coverage(df: pd.DataFrame) -> dict:
    missing_columns = sorted(REQUIRED_RESULT_COLUMNS.difference(df.columns))
    if missing_columns:
        return {"valid_schema": False, "missing_columns": missing_columns}

    pairs = df[["K_bp", "K_rp", "repeat"]].dropna().copy()
    pairs["K_bp"] = pairs["K_bp"].astype(int)
    pairs["K_rp"] = pairs["K_rp"].astype(int)
    pairs["repeat"] = pairs["repeat"].astype(int)
    pairs = pairs.drop_duplicates(["K_bp", "K_rp", "repeat"])
    in_grid = pairs[pairs["K_bp"].isin(K_BP_LIST) & pairs["K_rp"].isin(K_RP_LIST)]
    repeat_counts = in_grid.groupby(["K_bp", "K_rp"])["repeat"].nunique()
    complete = (
        len(repeat_counts) == len(K_BP_LIST) * len(K_RP_LIST)
        and not repeat_counts.empty
        and int(repeat_counts.min()) >= int(N_REPEATS)
    )
    return {
        "valid_schema": True,
        "rows": int(len(df)),
        "unique_k_repeat_rows": int(len(pairs)),
        "grid_points": int(len(repeat_counts)),
        "min_repeats_per_grid_point": int(repeat_counts.min()) if not repeat_counts.empty else 0,
        "max_repeats_per_grid_point": int(repeat_counts.max()) if not repeat_counts.empty else 0,
        "complete": bool(complete),
    }


def _candidate_result_paths() -> list[Path]:
    candidates = [RAW_PARQUET, RAW_CSV]
    candidates.extend(_resolve_bp_rp_grid_path(path) for path in ALTERNATE_RESULT_PATHS)

    unique = []
    seen = set()
    for candidate in candidates:
        key = str(candidate.resolve()) if candidate.exists() else str(candidate)
        if key not in seen:
            unique.append(candidate)
            seen.add(key)
    return unique


def _load_chunked_bp_rp_grid_results():
    for chunk_dir in BP_RP_GRID_CHUNK_DIRS:
        chunk_dir = _resolve_bp_rp_grid_path(chunk_dir)
        chunk_files = sorted(chunk_dir.glob("bp_rp_grid_r*_chunk*.parquet"))
        if not chunk_files:
            continue
        print(f"Loading chunked BP/RP grid results from {chunk_dir} ({len(chunk_files)} files)", flush=True)
        parts = []
        for index, path in enumerate(chunk_files, start=1):
            parts.append(pd.read_parquet(path))
            if index == 1 or index % SHARD_PROGRESS_EVERY == 0 or index == len(chunk_files):
                print(f"  loaded {index}/{len(chunk_files)} chunks", flush=True)
        df = pd.concat(parts, ignore_index=True).drop_duplicates(["K_bp", "K_rp", "repeat"], keep="last")
        coverage = _bp_rp_grid_coverage(df)
        return df, chunk_dir, coverage
    return None, None, None


def load_precomputed_bp_rp_grid_results():
    if not USE_PRECOMPUTED_BP_RP_GRID_RESULTS:
        return None, None, False

    partial_candidate = None
    coverage_columns = sorted(REQUIRED_RESULT_COLUMNS)
    for candidate in _candidate_result_paths():
        if not candidate.exists():
            continue
        print("Checking BP/RP grid result table:", candidate, flush=True)
        try:
            coverage_df = _read_bp_rp_grid_result(candidate, columns=coverage_columns)
        except Exception as exc:
            print(f"  rejected: could not read coverage columns ({type(exc).__name__}: {exc})", flush=True)
            continue

        coverage = _bp_rp_grid_coverage(coverage_df)
        if not coverage.get("valid_schema", False):
            print(f"  rejected: missing columns {coverage.get('missing_columns')}", flush=True)
            continue
        print(
            "  coverage: "
            f"rows={coverage['rows']}, grid={coverage['grid_points']}, "
            f"repeats per grid point={coverage['min_repeats_per_grid_point']}-{coverage['max_repeats_per_grid_point']}",
            flush=True,
        )

        if coverage["complete"] or ALLOW_PARTIAL_PRECOMPUTED_RESULTS:
            try:
                df = _read_bp_rp_grid_result(candidate)
            except Exception as exc:
                print(f"  rejected: coverage is readable but full table read failed ({type(exc).__name__}: {exc})", flush=True)
                continue
            if coverage["complete"]:
                return df, candidate, True
            if partial_candidate is None:
                partial_candidate = (df, candidate, False)

    if partial_candidate is not None:
        return partial_candidate

    chunk_df, chunk_path, chunk_coverage = _load_chunked_bp_rp_grid_results()
    if chunk_df is not None:
        print(
            "  chunk coverage: "
            f"rows={chunk_coverage['rows']}, grid={chunk_coverage['grid_points']}, "
            f"repeats per grid point={chunk_coverage['min_repeats_per_grid_point']}-{chunk_coverage['max_repeats_per_grid_point']}",
            flush=True,
        )
        if chunk_coverage["complete"] or ALLOW_PARTIAL_PRECOMPUTED_RESULTS:
            return chunk_df, chunk_path, bool(chunk_coverage["complete"])

    return None, None, False


res_bp_rp_grid, BP_RP_GRID_ACTIVE_RESULT_PATH, BP_RP_GRID_RESULT_IS_COMPLETE = load_precomputed_bp_rp_grid_results()

if res_bp_rp_grid is not None:
    print("Using BP/RP grid result table:", BP_RP_GRID_ACTIVE_RESULT_PATH, flush=True)
    print("Result rows:", res_bp_rp_grid.shape, flush=True)
    if BP_RP_GRID_RESULT_IS_COMPLETE:
        active_path = Path(BP_RP_GRID_ACTIVE_RESULT_PATH)
        if not RAW_PARQUET.exists() or active_path.resolve() != RAW_PARQUET.resolve():
            res_bp_rp_grid.to_parquet(RAW_PARQUET, index=False)
            print("Saved canonical BP/RP grid parquet:", RAW_PARQUET)
        else:
            print("Canonical BP/RP grid parquet is already available:", RAW_PARQUET)
        if WRITE_BP_RP_GRID_CSV:
            res_bp_rp_grid.to_csv(RAW_CSV, index=False)
            print("Saved canonical BP/RP grid CSV:", RAW_CSV)
    else:
        partial_parquet = OUT_DIR / RAW_OUTPUT_BASENAME.replace("_raw.csv", "_partial_raw.parquet")
        partial_csv = OUT_DIR / RAW_OUTPUT_BASENAME.replace("_raw.csv", "_partial_raw.csv")
        res_bp_rp_grid.to_parquet(partial_parquet, index=False)
        BP_RP_GRID_ACTIVE_RESULT_PATH = partial_parquet
        print("Loaded partial BP/RP grid results. Reporting cells may be used, but full-sweep conclusions require more repeats.")
        print("Saved partial BP/RP grid parquet:", partial_parquet)
        if WRITE_BP_RP_GRID_CSV:
            res_bp_rp_grid.to_csv(partial_csv, index=False)
            print("Saved partial BP/RP grid CSV:", partial_csv)
elif not RUN_BP_RP_GRID_SWEEP:
    raise RuntimeError(
        "No usable precomputed BP/RP grid result table was found. "
        "Set RUN_BP_RP_GRID_SWEEP = True before this cell to run the BP/RP grid sweep."
    )
else:
    # Run only missing repeats
    to_run = [sp for sp in splits if not done_rep(sp["repeat"])]
    print("Repeats total:", len(splits), "| already done:", len(splits) - len(to_run), "| to run:", len(to_run))
    pairs_per_repeat = len(K_BP_LIST) * len(K_RP_LIST)
    pairs_to_run = sum(max(0, pairs_per_repeat - done_pair_count(int(sp["repeat"]))) for sp in to_run)
    print("Pairs to run:", pairs_to_run, "| per repeat:", pairs_per_repeat)

    @contextmanager
    def tqdm_joblib(tqdm_object):
        class TqdmBatchCompletionCallback(joblib.parallel.BatchCompletionCallBack):
            def __call__(self, *args, **kwargs):
                tqdm_object.update(n=self.batch_size)
                return super().__call__(*args, **kwargs)
        old_callback = joblib.parallel.BatchCompletionCallBack
        joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback
        try:
            yield tqdm_object
        finally:
            joblib.parallel.BatchCompletionCallBack = old_callback
            tqdm_object.close()

    PAIR_PBAR_LOCK = threading.Lock()
    PAIR_PBAR = None

    if to_run:
        pair_bar = tqdm(total=pairs_to_run, desc="BP/RP grid pairs", unit="pair", position=0, dynamic_ncols=True)
        try:
            PAIR_PBAR = pair_bar
            with tqdm_joblib(tqdm(total=len(to_run), desc="BP/RP grid repeats", unit="repeat", position=1, dynamic_ncols=True)):
                shard_paths = Parallel(
                    n_jobs=N_JOBS,
                    backend="threading",
                    batch_size=1,
                    pre_dispatch="2*n_jobs",
                )(
                    delayed(run_one_repeat_bp_rp_grid)(sp) for sp in to_run
                )
        finally:
            PAIR_PBAR = None
            pair_bar.close()
    else:
        shard_paths = []

    # Merge all repeat shards
    files = sorted(BP_RP_GRID_RUNS_DIR.glob("bp_rp_grid_r*.parquet"))
    if not files:
        raise RuntimeError("No BP/RP grid repeat shards found. Let at least one repeat finish first.")
    res_bp_rp_grid = pd.concat([pd.read_parquet(p) for p in files], ignore_index=True)

    # Save merged outputs
    out_parquet = OUT_DIR / "truncation_knn_bp_rp_grid_raw.parquet"
    out_csv     = OUT_DIR / "truncation_knn_bp_rp_grid_raw.csv"
    res_bp_rp_grid.to_parquet(out_parquet, index=False)
    if WRITE_BP_RP_GRID_CSV:
        res_bp_rp_grid.to_csv(out_csv, index=False)

    print("Saved:", out_parquet)
    if WRITE_BP_RP_GRID_CSV:
        print("Saved:", out_csv)
    print("Rows:", res_bp_rp_grid.shape[0], "Cols:", res_bp_rp_grid.shape[1])
    display(res_bp_rp_grid.head(5))

    BP_RP_GRID_ACTIVE_RESULT_PATH = RAW_PARQUET
    BP_RP_GRID_RESULT_IS_COMPLETE = True


## Summaries and standardized BP/RP grid plots


In [ ]:

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy import stats
from IPython.display import display

if "res_bp_rp_grid" not in globals():
    if "BP_RP_GRID_ACTIVE_RESULT_PATH" in globals() and Path(BP_RP_GRID_ACTIVE_RESULT_PATH).exists():
        res_bp_rp_grid = _read_bp_rp_grid_result(Path(BP_RP_GRID_ACTIVE_RESULT_PATH))
    elif RAW_PARQUET.exists():
        res_bp_rp_grid = pd.read_parquet(RAW_PARQUET)
        BP_RP_GRID_ACTIVE_RESULT_PATH = RAW_PARQUET
    elif RAW_CSV.exists():
        res_bp_rp_grid = pd.read_csv(RAW_CSV)
        BP_RP_GRID_ACTIVE_RESULT_PATH = RAW_CSV
    else:
        raise FileNotFoundError("Run the BP/RP grid result-loading cell before building summaries and plots.")

MODEL_LABEL = globals().get("MODEL_LABEL", "BP/RP grid model")
SUMMARY_OUTPUT_BASENAME = RAW_OUTPUT_BASENAME.replace("_raw.csv", "_summary_by_pair.csv")
SHOW_BP_RP_GRID_FIGURES = bool(globals().get("SHOW_BP_RP_GRID_FIGURES", True))
WRITE_BP_RP_GRID_HTML = bool(globals().get("WRITE_BP_RP_GRID_HTML", True))
SUMMARY_CSV = OUT_DIR / SUMMARY_OUTPUT_BASENAME
SUMMARY_PARQUET = OUT_DIR / SUMMARY_OUTPUT_BASENAME.replace(".csv", ".parquet")

BASE_METRICS = [
    "runtime_s",
    "test_PR_AUC", "test_ROC_AUC", "test_LogLoss", "test_Brier",
    "youden_test_Precision", "youden_test_Sensitivity", "youden_test_Specificity", "youden_val_thr",
    "f1_test_Precision", "f1_test_Sensitivity", "f1_test_Specificity", "f1_val_thr",
]
SUMMARY_METRICS = [metric for metric in BASE_METRICS if metric in res_bp_rp_grid.columns]


def _metric_label(metric: str) -> str:
    labels = {
        "test_PR_AUC": "PR AUC",
        "test_ROC_AUC": "ROC AUC",
        "test_LogLoss": "Log loss",
        "test_Brier": "Brier score",
        "youden_test_Precision": "Youden precision",
        "youden_test_Sensitivity": "Youden sensitivity",
        "youden_test_Specificity": "Youden specificity",
        "f1_test_Precision": "F1 precision",
        "f1_test_Sensitivity": "F1 sensitivity",
        "f1_test_Specificity": "F1 specificity",
        "runtime_s": "Runtime (s)",
    }
    return labels.get(metric, metric.replace("_", " "))


def _metric_lower_is_better(metric: str) -> bool:
    metric_lower = metric.lower()
    return "loss" in metric_lower or "brier" in metric_lower or metric_lower == "runtime_s"


def q05(values):
    return float(np.nanquantile(values, 0.05))


def q50(values):
    return float(np.nanquantile(values, 0.50))


def q95(values):
    return float(np.nanquantile(values, 0.95))


def summarize_bp_rp_grid_metrics(df: pd.DataFrame, metrics: list[str]) -> pd.DataFrame:
    summaries = []
    for metric in metrics:
        values = df[["K_bp", "K_rp", metric]].dropna().copy()
        if values.empty:
            continue
        summary = (
            values.groupby(["K_bp", "K_rp"], as_index=False)[metric]
            .agg(mean="mean", std="std", n="count", q05=q05, q50=q50, q95=q95)
        )
        summary["std"] = summary["std"].fillna(0.0)
        summary["se"] = summary["std"] / np.sqrt(summary["n"].clip(lower=1))
        summary["tcrit"] = summary["n"].apply(lambda n: stats.t.ppf(0.975, int(n) - 1) if int(n) > 1 else 0.0)
        summary["ci95_lo"] = summary["mean"] - summary["tcrit"] * summary["se"]
        summary["ci95_hi"] = summary["mean"] + summary["tcrit"] * summary["se"]
        summary.insert(2, "metric", metric)
        summaries.append(summary[["K_bp", "K_rp", "metric", "mean", "std", "n", "se", "ci95_lo", "ci95_hi", "q05", "q50", "q95"]])
    return pd.concat(summaries, ignore_index=True) if summaries else pd.DataFrame()


summary_bp_rp_grid = summarize_bp_rp_grid_metrics(res_bp_rp_grid, SUMMARY_METRICS)
if summary_bp_rp_grid.empty:
    raise RuntimeError("No metric summaries were generated from the BP/RP grid result table.")

summary_bp_rp_grid.to_parquet(SUMMARY_PARQUET, index=False)
summary_bp_rp_grid.to_csv(SUMMARY_CSV, index=False)
print("Saved BP/RP grid summary outputs:")
print(" -", SUMMARY_PARQUET)
print(" -", SUMMARY_CSV)
display(summary_bp_rp_grid.head(10))


def _metric_summary(metric: str) -> pd.DataFrame:
    if metric not in summary_bp_rp_grid["metric"].unique():
        raise KeyError(f"Metric '{metric}' is not available. Available metrics: {sorted(summary_bp_rp_grid['metric'].unique())}")
    return summary_bp_rp_grid[summary_bp_rp_grid["metric"] == metric].copy()


def _pivot_metric(metric: str, value_col: str) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    metric_summary = _metric_summary(metric)
    pivot = metric_summary.pivot(index="K_rp", columns="K_bp", values=value_col).sort_index().sort_index(axis=1)
    x_values = pivot.columns.to_numpy(int)
    y_values = pivot.index.to_numpy(int)
    z_values = pivot.to_numpy(float)
    return x_values, y_values, z_values


def _custom_grid(metric: str) -> np.ndarray:
    columns = ["mean", "ci95_lo", "ci95_hi", "std", "n", "q05", "q95"]
    grids = [_pivot_metric(metric, column)[2] for column in columns]
    return np.stack(grids, axis=-1)


def _write_plot(fig: go.Figure, filename: str):
    if not WRITE_BP_RP_GRID_HTML:
        return
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    path = FIGURE_DIR / filename
    fig.write_html(path, include_plotlyjs="cdn")
    print("Saved figure:", path)


def _ranked_grid_points(metric: str, value_col: str = "mean", top_n: int = 12) -> pd.DataFrame:
    metric_summary = _metric_summary(metric)
    ascending = _metric_lower_is_better(metric)
    return (
        metric_summary.sort_values(value_col, ascending=ascending)
        .head(int(top_n))
        .assign(rank=lambda frame: np.arange(1, len(frame) + 1))
        [["rank", "K_bp", "K_rp", value_col, "ci95_lo", "ci95_hi", "std", "n"]]
    )


def plot_bp_rp_grid_heatmap(metric: str = "test_PR_AUC", value_col: str = "mean", top_n: int = 12,
                        colorscale: str = "Viridis", show_top_table: bool = True) -> pd.DataFrame:
    x_values, y_values, z_values = _pivot_metric(metric, value_col)
    custom = _custom_grid(metric)
    metric_label = _metric_label(metric)
    top = _ranked_grid_points(metric, value_col=value_col, top_n=top_n)

    fig = go.Figure()
    fig.add_trace(go.Heatmap(
        x=x_values,
        y=y_values,
        z=z_values,
        customdata=custom,
        colorscale=colorscale,
        colorbar=dict(title=f"{metric_label}<br>{value_col}"),
        hovertemplate=(
            "K_bp=%{x}<br>K_rp=%{y}<br>"
            "mean=%{customdata[0]:.5f}<br>"
            "95% CI=[%{customdata[1]:.5f}, %{customdata[2]:.5f}]<br>"
            "std=%{customdata[3]:.5f}<br>"
            "n=%{customdata[4]:.0f}<br>"
            "q05=%{customdata[5]:.5f}<br>"
            "q95=%{customdata[6]:.5f}<extra></extra>"
        ),
    ))

    if not top.empty:
        fig.add_trace(go.Scatter(
            x=top["K_bp"],
            y=top["K_rp"],
            mode="markers+text",
            text=top["rank"].astype(str),
            textposition="middle center",
            marker=dict(size=18, color="rgba(220, 20, 60, 0.65)", line=dict(color="white", width=1)),
            name="Top ranked",
            hovertemplate="rank=%{text}<br>K_bp=%{x}<br>K_rp=%{y}<extra></extra>",
        ))

    fig.update_layout(
        title=f"{MODEL_LABEL}: {metric_label} across independent BP/RP truncation ({value_col})",
        width=880,
        height=760,
        margin=dict(l=60, r=30, t=80, b=60),
        xaxis=dict(title="K_bp retained", dtick=5),
        yaxis=dict(title="K_rp retained", dtick=5, scaleanchor="x", scaleratio=1),
        plot_bgcolor="white",
    )
    if SHOW_BP_RP_GRID_FIGURES:
        fig.show()
    _write_plot(fig, f"{MODEL_LABEL.lower()}_bp_rp_grid_{metric}_{value_col}_heatmap.html")

    if show_top_table:
        print(f"Top {len(top)} grid points for {metric_label} ({'lower is better' if _metric_lower_is_better(metric) else 'higher is better'}):")
        display(top)
    return top


def plot_bp_rp_grid_surface(metric: str = "test_PR_AUC", value_col: str = "mean", colorscale: str = "Viridis") -> go.Figure:
    x_values, y_values, z_values = _pivot_metric(metric, value_col)
    custom = _custom_grid(metric)
    metric_label = _metric_label(metric)

    fig = go.Figure()
    fig.add_trace(go.Surface(
        x=x_values,
        y=y_values,
        z=z_values,
        customdata=custom,
        colorscale=colorscale,
        colorbar=dict(title=f"{metric_label}<br>{value_col}"),
        hovertemplate=(
            "K_bp=%{x}<br>K_rp=%{y}<br>"
            "mean=%{customdata[0]:.5f}<br>"
            "95% CI=[%{customdata[1]:.5f}, %{customdata[2]:.5f}]<br>"
            "std=%{customdata[3]:.5f}<br>"
            "n=%{customdata[4]:.0f}<extra></extra>"
        ),
    ))

    top = _ranked_grid_points(metric, value_col=value_col, top_n=1)
    if not top.empty:
        best = top.iloc[0]
        fig.add_trace(go.Scatter3d(
            x=[best["K_bp"]], y=[best["K_rp"]], z=[best[value_col]],
            mode="markers+text",
            text=["best"],
            textposition="top center",
            marker=dict(size=7, color="crimson", symbol="diamond"),
            name="Best",
        ))

    fig.update_layout(
        title=f"{MODEL_LABEL}: {metric_label} surface across independent BP/RP truncation ({value_col})",
        height=760,
        margin=dict(l=0, r=0, t=80, b=0),
        scene=dict(
            xaxis=dict(title="K_bp retained", dtick=5),
            yaxis=dict(title="K_rp retained", dtick=5),
            zaxis=dict(title=f"{metric_label} {value_col}"),
            aspectmode="cube",
        ),
    )
    if SHOW_BP_RP_GRID_FIGURES:
        fig.show()
    _write_plot(fig, f"{MODEL_LABEL.lower()}_bp_rp_grid_{metric}_{value_col}_surface.html")
    return fig


DEFAULT_BP_RP_GRID_METRIC = globals().get("DEFAULT_BP_RP_GRID_METRIC", "test_PR_AUC")
DEFAULT_BP_RP_GRID_VALUE = globals().get("DEFAULT_BP_RP_GRID_VALUE", "mean")
plot_bp_rp_grid_heatmap(DEFAULT_BP_RP_GRID_METRIC, value_col=DEFAULT_BP_RP_GRID_VALUE, top_n=12)
plot_bp_rp_grid_surface(DEFAULT_BP_RP_GRID_METRIC, value_col=DEFAULT_BP_RP_GRID_VALUE)
